# Results

In this notebook, existing models are loaded and then evaluated on the correct data to generate results.

For this to run, all datasets must be setup and checkpoints must be stored in the right places.

Crossval checkpoints are expected in ```"../checkpoints/fold{0..7}"```.

Final checkpoints are expected in ```"../checkpoints/final{0..2}"```.

In [ ]:
from pathlib import Path

import torch
import pandas as pd
from pandas import DataFrame

from explore_this.dataset.dataset import BeatDataModule
from explore_this.model.pl_module import PLExploreThis


In [2]:
from pytorch_lightning import Trainer, seed_everything
device = "cuda:0"

def datamodule_setup(checkpoint, num_workers, datasplit):
    # Load the datamodule
    print("Creating datamodule")
    data_dir = "../data"
    datamodule_hparams = checkpoint["datamodule_hyper_parameters"]
    # update the hparams with the ones from the arguments
    if num_workers is not None:
        datamodule_hparams["num_workers"] = num_workers
    datamodule_hparams["predict_datasplit"] = datasplit
    datamodule_hparams["data_dir"] = data_dir
    datamodule = BeatDataModule(**datamodule_hparams)
    datamodule.setup(stage="predict")
    return datamodule

# Cross validation

In [ ]:
def fold_path(i):
    return Path(f"../checkpoints/fold{i}")

def namify_piece_path(s):
    return "/".join(s.split("/")[:-1])

def switch_to_viterbi(model: PLExploreThis):
    model.model.subgrid_processor.mask_method = "viterbi"
    model.model.subgrid_processor.viterbi_transition_prob = 0.3

def parse_value(s):
    try:
        return float(s)
    except ValueError:
        return float(s[7:-1])
def humanize_number(x):
    return f"{x*100:.1f}"
    

In [ ]:
def metrics_for_fold(fold: int, model_adjustor=None):
    checkpoint_path = fold_path(fold)
    model = PLExploreThis.load_from_checkpoint(checkpoint_path=checkpoint_path, map_location=device)
    model = model.eval()
    if model_adjustor is not None:
        model_adjustor(model)

    dm = datamodule_setup(
        torch.load(checkpoint_path, map_location=device), 8, "val"
    )

    trainer = Trainer(accelerator="gpu", devices=[0])
    output = trainer.predict(model, datamodule=dm)

    res = {}
    for metrics, pred, dataset, spectpath in output:
        res[namify_piece_path(spectpath[0])] = metrics 
    return res

## Crossval threshold

In [40]:
seed_everything(0, workers=True)
metrics_df_path = "./output/cross_validation_metrics.csv"
try:
    metrics_df = pd.read_csv(metrics_df_path, index_col=0)
except FileNotFoundError:
    print("Rerunning")
    all_metrics_dict = {}

    for fold in range(8):
        all_metrics_dict.update(metrics_for_fold(fold))

    metrics_df = DataFrame.from_dict(all_metrics_dict, orient="index")
    metrics_df = metrics_df.map(lambda x: parse_value(x))
    metrics_df.to_csv(metrics_df_path)



Seed set to 0


In [41]:
metrics_df["dataset"]=metrics_df.index.map(lambda s: s.split("/")[0])
metrics_df.groupby("dataset").mean()

,grid_recall,grid_downbeat_recall,grid_F-measure,grid_AMLt,grid_CMLt,subgrid_bF-measure,subgrid_bRecall,subgrid_bAMLt,subgrid_bCMLt,subgrid_dbF-measure,subgrid_dbRecall,subgrid_dbAMLt,subgrid_dbCMLt
dataset,,,,,,,,,,,,,
asap,0.808688,0.879390,0.651922,0.563190,0.341892,0.723270,0.737257,0.578989,0.503952,0.588247,0.600161,0.452533,0.311854
ballroom,0.967566,0.976010,0.876818,0.898793,0.720457,0.953545,0.964512,0.942430,0.927972,0.948238,0.954262,0.962799,0.949572
beatles,0.972551,0.973996,0.864859,0.890151,0.685579,0.929670,0.944677,0.909999,0.848255,0.881963,0.884507,0.870053,0.777737
candombe,0.992634,0.991117,0.984549,0.977172,0.977172,0.990477,0.992634,0.989102,0.989102,0.990451,0.990520,0.995745,0.995745
filosax,0.930254,0.992317,0.939560,0.915258,0.836031,0.950367,0.929856,0.937348,0.855473,0.985979,0.987381,0.986879,0.986879
groove_midi,0.923965,0.923738,0.826646,0.873637,0.610565,0.896869,0.900068,0.884575,0.829157,0.806356,0.807950,0.866329,0.774830
guitarset,0.893149,0.896645,0.822453,0.834443,0.672495,0.879472,0.884467,0.864140,0.849723,0.866153,0.868057,0.921507,0.896041
hainsworth,0.932686,0.950378,0.851495,0.908789,0.670495,0.896682,0.900011,0.900276,0.811997,0.801691,0.820761,0.818133,0.733979
harmonix,0.969091,0.971329,0.885881,0.926244,0.726370,0.943254,0.948868,0.923724,0.884828,0.901296,0.906215,0.901929,0.851649


## Threshold results

In [84]:
metrics_df.groupby("dataset")["subgrid_bCMLt"].mean().map(humanize_number)

dataset
asap           50.4
ballroom       92.8
beatles        84.8
candombe       98.9
filosax        85.5
groove_midi    82.9
guitarset      85.0
hainsworth     81.2
harmonix       88.5
hjdb           94.6
jaah           68.5
rwc            74.3
simac          72.7
smc            48.0
tapcorrect     82.2
Name: subgrid_bCMLt, dtype: str

In [85]:
metrics_df.groupby("dataset")["subgrid_dbCMLt"].mean().map(humanize_number)

dataset
asap           31.2
ballroom       95.0
beatles        77.8
candombe       99.6
filosax        98.7
groove_midi    77.5
guitarset      89.6
hainsworth     73.4
harmonix       85.2
hjdb           94.2
jaah           78.8
rwc            72.3
simac           0.0
smc             0.0
tapcorrect     79.3
Name: subgrid_dbCMLt, dtype: str

In [89]:
humanize_number(metrics_df["subgrid_bCMLt"].mean())

'79.2'

In [90]:
humanize_number(metrics_df[~metrics_df["dataset"].isin(("simac", "smc"))]["subgrid_dbCMLt"].mean())

'78.4'

## Crossval viterbi

In [49]:
seed_everything(0, workers=True)
viterbi_metrics_df_path = "./output/cross_validation_metrics_viterbi_03.csv"
try:
    viterbi_metrics_df = pd.read_csv(viterbi_metrics_df_path, index_col=0)
except FileNotFoundError:
    all_metrics_dict = {}

    for fold in range(8):
        all_metrics_dict.update(metrics_for_fold(fold, model_adjustor=switch_to_viterbi))

    viterbi_metrics_df = DataFrame.from_dict(all_metrics_dict, orient="index")
    viterbi_metrics_df = viterbi_metrics_df.map(lambda x: parse_value(x))
    viterbi_metrics_df.to_csv(viterbi_metrics_df_path)

Seed set to 0


In [54]:
viterbi_metrics_df["dataset"]=viterbi_metrics_df.index.map(lambda s: s.split("/")[0])
viterbi_metrics_df

,grid_recall,grid_downbeat_recall,grid_F-measure,grid_AMLt,grid_CMLt,subgrid_bF-measure,subgrid_bRecall,subgrid_bAMLt,subgrid_bCMLt,subgrid_dbF-measure,subgrid_dbRecall,subgrid_dbAMLt,subgrid_dbCMLt,dataset
asap/Bach_Fugue_bwv_862_Song04M,0.9928,1.0000,0.6357,0.921502,0.000000,0.9395,0.9565,0.902098,0.902098,0.3178,0.5000,0.410959,0.000000,asap
asap/Bach_Fugue_bwv_863_LeeN01M,1.0000,1.0000,0.6465,0.946269,0.000000,0.9877,1.0000,0.969512,0.969512,0.5647,0.6000,0.422222,0.422222,asap
asap/Bach_Fugue_bwv_863_Shychko01M,0.9812,0.9750,0.6347,0.923304,0.000000,0.9694,0.9812,0.957831,0.957831,0.3059,0.3250,0.377778,0.222222,asap
asap/Bach_Fugue_bwv_863_TongB01M,0.9875,0.9750,0.6320,0.917647,0.000000,0.9632,0.9812,0.951807,0.951807,0.3958,0.4750,0.250000,0.250000,asap
asap/Bach_Fugue_bwv_868_GonzalezJ05M,0.9848,1.0000,0.6357,0.931408,0.000000,0.9630,0.9848,0.942029,0.942029,0.7246,0.7576,0.638889,0.638889,asap
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
tapcorrect/084_youtube_zH9sXggRlxc,0.9873,0.9915,0.9588,0.938000,0.938000,0.9759,0.9873,0.971014,0.971014,0.9791,0.9915,0.975207,0.975207,tapcorrect
tapcorrect/088_youtube_D1ACUFvxAGQ,0.9340,0.9815,0.6130,0.917051,0.002304,0.7659,0.9104,0.452055,0.452055,0.5238,0.6111,0.402778,0.402778,tapcorrect
tapcorrect/089_youtube_Ghg70fI6HLI,1.0000,1.0000,0.6547,0.969697,0.000000,0.9909,1.0000,0.982014,0.982014,0.9928,1.0000,0.985714,0.985714,tapcorrect
tapcorrect/094_youtube_FrGARptU-kQ,0.9806,0.9841,0.9727,0.965898,0.965898,0.9776,0.9796,0.977665,0.977665,0.6600,0.6614,0.630952,0.630952,tapcorrect


## Viterbi results

In [91]:
viterbi_metrics_df.groupby("dataset")["subgrid_bCMLt"].mean().map(humanize_number)

dataset
asap           50.1
ballroom       90.6
beatles        83.3
candombe       98.3
filosax        84.9
groove_midi    83.4
guitarset      84.5
hainsworth     81.4
harmonix       87.9
hjdb           94.6
jaah           68.1
rwc            73.3
simac          72.6
smc            48.5
tapcorrect     79.8
Name: subgrid_bCMLt, dtype: str

In [92]:
viterbi_metrics_df.groupby("dataset")["subgrid_dbCMLt"].mean().map(humanize_number)

dataset
asap           28.6
ballroom       92.5
beatles        76.1
candombe       99.3
filosax        97.7
groove_midi    76.8
guitarset      86.9
hainsworth     73.6
harmonix       84.5
hjdb           94.3
jaah           78.3
rwc            70.6
simac           0.0
smc             0.0
tapcorrect     74.8
Name: subgrid_dbCMLt, dtype: str

In [93]:
humanize_number(viterbi_metrics_df["subgrid_bCMLt"].mean())

'78.5'

In [94]:
humanize_number(viterbi_metrics_df[~viterbi_metrics_df["dataset"].isin(("simac", "smc"))]["subgrid_dbCMLt"].mean())

'76.9'

## Grid recall

In [95]:
metrics_df.groupby("dataset")["grid_recall"].mean().map(humanize_number)

dataset
asap           80.9
ballroom       96.8
beatles        97.3
candombe       99.3
filosax        93.0
groove_midi    92.4
guitarset      89.3
hainsworth     93.3
harmonix       96.9
hjdb           95.8
jaah           84.1
rwc            90.1
simac          87.6
smc            69.1
tapcorrect     96.6
Name: grid_recall, dtype: str

In [97]:
humanize_number(metrics_df["grid_recall"].mean())


'91.1'

# Final test!

In [54]:
def seed_path(seed: int) -> Path:
    return Path(f"../checkpoints/final{seed}")

In [ ]:
def metrics_for_seed(seed: int, model_adjustor=None):
    checkpoint_path = seed_path(seed)
    model = PLExploreThis.load_from_checkpoint(checkpoint_path=checkpoint_path, map_location=device)
    model = model.eval()
    if model_adjustor is not None:
        model_adjustor(model)
    dm = datamodule_setup(
        torch.load(checkpoint_path, map_location=device), 8, "test"
    )

    trainer = Trainer(accelerator="gpu", devices=[0])
    output = trainer.predict(model, datamodule=dm)

    res = {}
    for metrics, pred, dataset, spectpath in output:
        res[namify_piece_path(spectpath[0])] = metrics 
    return res

In [67]:
seed_everything(0, workers=True)
final_metrics_df_path = "./output/final_metrics.csv"
try:
    final_metrics_df = pd.read_csv(final_metrics_df_path, index_col=(0,1))
except FileNotFoundError:
    all_models_dict = {}

    for seed in range(3):
        all_models_dict[seed] = metrics_for_seed(seed)

    final_metrics_df = pd.concat(
        {
            model: pd.DataFrame.from_dict(data, orient="index")
            for model, data in all_models_dict.items()
        },
        names=["model", "excerpt"],
    )
    final_metrics_df = final_metrics_df.map(lambda x: parse_value(x))
    final_metrics_df.to_csv(final_metrics_df_path)

Seed set to 0


## Results

In [98]:
def summarize_results(mutli_model_df):
    for metric in (
        "subgrid_bF-measure",
        "subgrid_bCMLt",
        "subgrid_bAMLt",
        "subgrid_dbF-measure",
        "subgrid_dbCMLt",
        "subgrid_dbAMLt",
        "grid_recall",
        "grid_AMLt"
    ):
        print(metric)
        val = mutli_model_df.groupby("model")[metric].mean()
        print(f"{humanize_number(val.mean())}+-{humanize_number(val.std())}")

In [99]:
summarize_results(final_metrics_df)

subgrid_bF-measure
87.9+-0.8
subgrid_bCMLt
79.8+-1.2
subgrid_bAMLt
90.3+-0.1
subgrid_dbF-measure
77.3+-0.9
subgrid_dbCMLt
72.4+-0.7
subgrid_dbAMLt
85.1+-0.6
grid_recall
90.5+-2.0
grid_AMLt
89.6+-0.6


In [77]:
seed_everything(0, workers=True)
final_metrics_viterbi_df_path = "./output/final_metrics_viterbi_03.csv"
try:
    final_metrics_viterbi_df = pd.read_csv(final_metrics_viterbi_df_path, index_col=(0,1))
except FileNotFoundError:
    all_models_dict = {}

    for seed in range(3):
        all_models_dict[seed] = metrics_for_seed(seed, model_adjustor=switch_to_viterbi)

    final_metrics_viterbi_df = pd.concat(
        {
            model: pd.DataFrame.from_dict(data, orient="index")
            for model, data in all_models_dict.items()
        },
        names=["model", "excerpt"],
    )
    final_metrics_viterbi_df = final_metrics_viterbi_df.map(lambda x: parse_value(x))
    final_metrics_viterbi_df.to_csv(final_metrics_viterbi_df_path)

Seed set to 0


In [100]:
summarize_results(final_metrics_viterbi_df)

subgrid_bF-measure
88.0+-0.7
subgrid_bCMLt
80.0+-1.1
subgrid_bAMLt
90.4+-0.1
subgrid_dbF-measure
77.4+-1.0
subgrid_dbCMLt
72.7+-0.8
subgrid_dbAMLt
86.2+-0.4
grid_recall
90.5+-2.0
grid_AMLt
89.6+-0.6
